# PartnerCopilot: experimento D separado
Reaproveita o B original e o baseline C; nao gera as primeiras respostas novamente.
Uma unica correcao por saida rejeitada, parser-v2, prompt-v2 congelado.
Ative GPU, Internet, HF_TOKEN e use kernel novo. O codigo de D precisa estar
commitado e enviado ao GitHub. Preserve v1/v2 e seus notebooks.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import torch
assert torch.cuda.is_available(), 'Ative GPU.'
print(torch.cuda.get_device_name(0))
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['EMBEDDING_DEVICE'] = 'cpu'
os.environ['DATA_EVAL_DIR'] = 'data/eval/human-v2'
repo = Path('/kaggle/working/partner-copilot-D')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/JoaoNordemann/partner-copilot.git', str(repo)], check=True)
os.chdir(repo)
assert Path('scripts/evaluate_generation_retry.py').is_file(), 'Envie o codigo de D ao GitHub antes de usar este notebook.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Localizar B e validar sem inferencia
Se B nao estiver mais no working, anexe seu JSONL salvo como entrada do notebook
e ajuste source para o caminho real em /kaggle/input. Nao use o JSONL derivado de C.

In [ ]:
source = Path('/kaggle/working/PartnerCopilot/v2/generation-kaggle-B-prompt-v2-parser-v1.jsonl')
assert source.is_file(), 'Anexe o JSONL de B e ajuste source.'
output = Path('/kaggle/working/PartnerCopilot/D/generation-kaggle-D.jsonl')
command = [sys.executable, 'scripts/evaluate_generation_retry.py', '--source', str(source), '--output', str(output)]
subprocess.run(command + ['--check-only'], check=True)

## Rodar D
Reutiliza o contexto original. Apenas as saidas rejeitadas recebem correcao.
Se uma chamada ficar indeterminada apos interrupcao, nao sera repetida na retomada.

In [ ]:
subprocess.run(command + ['--resume'], check=True)

## Relatorio e preservacao
Baixe os dois JSONLs (resultado e ledger) e o resumo antes de encerrar a sessao.
As duas metricas sao diagnosticas e nao substituem leitura das respostas corrigidas.

In [ ]:
print(output.with_suffix('.summary.json').read_text(encoding='utf-8'))
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
zip_path = Path('/kaggle/working/generation-kaggle-D.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for path in output.parent.iterdir():
        if path.is_file() and path.suffix != '.lock':
            archive.write(path, arcname=path.name)
display(FileLink(str(zip_path)))